# 10 - Introduction to AI Agents (Conceptual Foundations)

This notebook introduces the **conceptual foundations** of AI agents. We stay framework-free—no LangChain, no AutoGen, no CrewAI. The goal is to build mental models that will serve you regardless of which tools you eventually use.

By the end of this notebook, you will be able to:
- Define what an agent is and how it differs from a simple model call.
- Describe three canonical architectures (reactive, deliberative, hybrid).
- Explain how agents select and invoke tools.
- Distinguish short-term and long-term memory strategies.
- Sketch a basic planning and control loop.
- Recognize when multi-agent designs are useful.
- Propose evaluation strategies for agent behavior.
- Connect these ideas to the CV models from Part 1.

---
## 1. Setup

This notebook is **conceptual**—no heavy dependencies, no GPU, no API keys.

We only use Python's standard library plus a few helpers for illustration.

In [1]:
import json
import random
from typing import Dict, List, Any, Optional, Callable
from dataclasses import dataclass, field
from enum import Enum
import textwrap

# For reproducibility in examples
random.seed(42)

print("Setup complete. No external dependencies required.")

Setup complete. No external dependencies required.


---
## 2. What Is an Agent?

### 2.1 The Core Idea

An **agent** is a system that:
1. **Perceives** its environment (receives inputs, observations, messages).
2. **Decides** what action to take next.
3. **Acts** on that decision (calls a tool, sends a message, updates state).
4. **Observes** the result and repeats.

This **observe → decide → act → update** loop is the skeleton of any agent system.

```
┌─────────────────────────────────────────────────┐
│                  ENVIRONMENT                    │
└──────────────────────┬──────────────────────────┘
                       │ observation
                       ▼
              ┌────────────────┐
              │     AGENT      │
              │                │
              │  ┌──────────┐  │
              │  │ Perceive │  │
              │  └────┬─────┘  │
              │       ▼        │
              │  ┌──────────┐  │
              │  │  Decide  │  │
              │  └────┬─────┘  │
              │       ▼        │
              │  ┌──────────┐  │
              │  │   Act    │  │
              │  └────┬─────┘  │
              └───────┼────────┘
                      │ action
                      ▼
┌─────────────────────────────────────────────────┐
│                  ENVIRONMENT                    │
└─────────────────────────────────────────────────┘
```

### 2.2 Agent vs. Model Call

A **single model call** is:
- Input → Model → Output
- One shot, no loop, no memory of previous calls.

An **agent** adds:
- A **loop** that can run multiple iterations.
- **State** that persists across iterations.
- **Tools** that extend what the agent can do.
- **Decisions** about when to stop, what tool to call, what to do next.

| Aspect | Single Model Call | Agent |
|--------|-------------------|-------|
| Iterations | 1 | Many |
| Memory | None | Short-term, optionally long-term |
| Tools | None | Can call external functions |
| Stopping | Immediate | Decides when goal is reached |
| Complexity | Low | Higher |

### 2.3 A Minimal Agent in Python

Let's build the simplest possible agent structure to make the concept concrete.

In [2]:
@dataclass
class AgentState:
    """Minimal state for our conceptual agent."""
    observations: List[str] = field(default_factory=list)
    actions_taken: List[str] = field(default_factory=list)
    goal_reached: bool = False
    iteration: int = 0


def minimal_agent_loop(
    initial_observation: str,
    decide_fn: Callable[[AgentState], str],
    act_fn: Callable[[str, AgentState], str],
    is_done_fn: Callable[[AgentState], bool],
    max_iterations: int = 10
) -> AgentState:
    """
    A minimal agent loop demonstrating the core pattern.
    
    Args:
        initial_observation: Starting observation from the environment.
        decide_fn: Function that chooses the next action given state.
        act_fn: Function that executes an action and returns new observation.
        is_done_fn: Function that checks if we should stop.
        max_iterations: Safety limit to prevent infinite loops.
    
    Returns:
        Final agent state after the loop completes.
    """
    state = AgentState()
    state.observations.append(initial_observation)
    
    while not is_done_fn(state) and state.iteration < max_iterations:
        # DECIDE: Choose an action based on current state
        action = decide_fn(state)
        
        # ACT: Execute the action, get new observation
        new_observation = act_fn(action, state)
        
        # UPDATE: Record what happened
        state.actions_taken.append(action)
        state.observations.append(new_observation)
        state.iteration += 1
        
        print(f"Iteration {state.iteration}: Action='{action}' → Observation='{new_observation}'")
    
    return state

print("Minimal agent loop defined.")

Minimal agent loop defined.


### 2.4 Example: A Counting Agent

Let's run a trivial agent that counts to a target number. This shows the loop in action.

In [3]:
# A simple counting agent
TARGET = 5

def counting_decide(state: AgentState) -> str:
    """Decide to increment if we haven't reached target."""
    current = len(state.actions_taken)
    if current < TARGET:
        return "increment"
    return "done"

def counting_act(action: str, state: AgentState) -> str:
    """Execute the action and return observation."""
    if action == "increment":
        new_count = len(state.actions_taken) + 1
        return f"count is now {new_count}"
    return "goal reached"

def counting_is_done(state: AgentState) -> bool:
    """Check if we've reached the goal."""
    return len(state.actions_taken) >= TARGET

# Run the agent
print(f"Goal: Count to {TARGET}")
print("-" * 40)
final_state = minimal_agent_loop(
    initial_observation="count is 0",
    decide_fn=counting_decide,
    act_fn=counting_act,
    is_done_fn=counting_is_done
)
print("-" * 40)
print(f"Final state: {len(final_state.actions_taken)} actions taken")

Goal: Count to 5
----------------------------------------
Iteration 1: Action='increment' → Observation='count is now 1'
Iteration 2: Action='increment' → Observation='count is now 2'
Iteration 3: Action='increment' → Observation='count is now 3'
Iteration 4: Action='increment' → Observation='count is now 4'
Iteration 5: Action='increment' → Observation='count is now 5'
----------------------------------------
Final state: 5 actions taken


> **Key Insight**: The pattern is always the same—loop, decide, act, observe, update. What changes is the complexity of each function.

---
## 3. Agent Architectures

Different problems call for different agent designs. The three canonical architectures are:

### 3.1 Reactive Agents

**Reactive agents** respond directly to observations without maintaining internal state or planning ahead.

```
Observation → Condition-Action Rules → Action
```

**Characteristics:**
- Fast response (no reasoning delay)
- Simple to implement
- No memory between interactions
- Good for well-defined, stable environments

**Example:** A thermostat. If temperature < 20°C → turn on heater. No planning, no memory of previous temperatures.

**When to use:**
- Real-time systems where latency matters
- Simple, predictable environments
- When you can enumerate all relevant conditions

In [4]:
class ReactiveAgent:
    """A simple reactive agent with condition-action rules."""
    
    def __init__(self, rules: List[tuple]):
        """
        Args:
            rules: List of (condition_fn, action) tuples.
                   condition_fn takes observation and returns bool.
        """
        self.rules = rules
    
    def act(self, observation: Dict[str, Any]) -> str:
        """Select action based on first matching rule."""
        for condition_fn, action in self.rules:
            if condition_fn(observation):
                return action
        return "no_action"  # Default if no rule matches


# Example: Store entrance monitoring agent
store_rules = [
    (lambda obs: obs.get("people_count", 0) > 100, "send_crowding_alert"),
    (lambda obs: obs.get("people_count", 0) > 50, "log_high_traffic"),
    (lambda obs: obs.get("people_count", 0) > 0, "log_normal_traffic"),
]

reactive_agent = ReactiveAgent(store_rules)

# Test with different observations
test_observations = [
    {"people_count": 120},
    {"people_count": 75},
    {"people_count": 10},
    {"people_count": 0},
]

print("Reactive Agent Demo: Store Monitoring")
print("-" * 40)
for obs in test_observations:
    action = reactive_agent.act(obs)
    print(f"People: {obs['people_count']:3d} → Action: {action}")

Reactive Agent Demo: Store Monitoring
----------------------------------------
People: 120 → Action: send_crowding_alert
People:  75 → Action: log_high_traffic
People:  10 → Action: log_normal_traffic
People:   0 → Action: no_action


### 3.2 Deliberative Agents

**Deliberative agents** maintain an internal model of the world and plan before acting.

```
Observation → Update World Model → Plan → Select Action → Act
```

**Characteristics:**
- Maintain internal state (beliefs about the world)
- Can plan multiple steps ahead
- More computationally expensive
- Better for complex, changing environments

**Example:** A chess-playing agent that thinks several moves ahead.

**When to use:**
- Complex tasks requiring multi-step reasoning
- When you need to consider future consequences
- Environments where planning pays off

In [5]:
@dataclass
class WorldModel:
    """Internal model of the world state."""
    beliefs: Dict[str, Any] = field(default_factory=dict)
    goals: List[str] = field(default_factory=list)
    
    def update(self, observation: Dict[str, Any]):
        """Update beliefs based on observation."""
        self.beliefs.update(observation)


class DeliberativeAgent:
    """A deliberative agent that plans before acting."""
    
    def __init__(self, goals: List[str]):
        self.world_model = WorldModel(goals=goals)
        self.plan: List[str] = []
    
    def perceive(self, observation: Dict[str, Any]):
        """Update world model with new observation."""
        self.world_model.update(observation)
    
    def plan_actions(self) -> List[str]:
        """
        Generate a plan to achieve goals.
        (Simplified: in real systems this would be more sophisticated)
        """
        plan = []
        beliefs = self.world_model.beliefs
        
        # Example planning logic for inventory management
        if beliefs.get("inventory_low", False):
            plan.append("check_supplier_availability")
            plan.append("create_purchase_order")
            plan.append("notify_warehouse")
        
        if beliefs.get("high_demand_predicted", False):
            plan.append("increase_safety_stock")
            plan.append("alert_sales_team")
        
        self.plan = plan
        return plan
    
    def get_next_action(self) -> Optional[str]:
        """Get the next action from the plan."""
        if not self.plan:
            return None
        return self.plan.pop(0)


# Demo
agent = DeliberativeAgent(goals=["maintain_stock_levels", "minimize_costs"])

# Agent perceives the world
agent.perceive({
    "inventory_low": True,
    "high_demand_predicted": True,
    "current_stock": 50,
    "reorder_point": 100
})

# Agent plans
plan = agent.plan_actions()
print("Deliberative Agent Demo: Inventory Management")
print("-" * 40)
print(f"Generated plan with {len(plan)} steps:")
for i, action in enumerate(plan, 1):
    print(f"  {i}. {action}")

Deliberative Agent Demo: Inventory Management
----------------------------------------
Generated plan with 5 steps:
  1. check_supplier_availability
  2. create_purchase_order
  3. notify_warehouse
  4. increase_safety_stock
  5. alert_sales_team


### 3.3 Hybrid Agents

**Hybrid agents** combine reactive and deliberative layers.

```
                    ┌─────────────────┐
                    │  Deliberative   │  (slow, strategic)
                    │     Layer       │
                    └────────┬────────┘
                             │ goals/plans
                             ▼
Observation ──────► ┌─────────────────┐ ──────► Action
                    │    Reactive     │  (fast, tactical)
                    │     Layer       │
                    └─────────────────┘
```

**Characteristics:**
- Reactive layer handles immediate responses
- Deliberative layer sets goals and strategies
- Best of both worlds: fast reactions + intelligent planning
- More complex to implement and tune

**Example:** A self-driving car. Reactive layer handles immediate obstacle avoidance. Deliberative layer plans the route.

**When to use:**
- Real-world systems needing both speed and intelligence
- When some situations need immediate response, others need planning

In [6]:
class HybridAgent:
    """
    Hybrid agent combining reactive and deliberative layers.
    Reactive layer can override deliberative plan for urgent situations.
    """
    
    def __init__(self, reactive_rules: List[tuple], goals: List[str]):
        self.reactive = ReactiveAgent(reactive_rules)
        self.deliberative = DeliberativeAgent(goals)
    
    def act(self, observation: Dict[str, Any]) -> tuple:
        """
        Decide action. Reactive layer gets priority for urgent situations.
        Returns (action, layer_used).
        """
        # Check reactive layer first (for urgent situations)
        reactive_action = self.reactive.act(observation)
        
        if reactive_action.startswith("emergency") or reactive_action.startswith("send_"):
            return (reactive_action, "reactive")
        
        # Otherwise use deliberative layer
        self.deliberative.perceive(observation)
        self.deliberative.plan_actions()
        deliberative_action = self.deliberative.get_next_action()
        
        if deliberative_action:
            return (deliberative_action, "deliberative")
        
        return (reactive_action, "reactive-fallback")


# Example: Warehouse robot
emergency_rules = [
    (lambda obs: obs.get("obstacle_detected", False), "emergency_stop"),
    (lambda obs: obs.get("battery_critical", False), "emergency_return_to_dock"),
    (lambda obs: True, "continue"),  # Default
]

hybrid_robot = HybridAgent(
    reactive_rules=emergency_rules,
    goals=["fulfill_orders", "maintain_battery"]
)

# Test scenarios
scenarios = [
    {"obstacle_detected": True, "battery_level": 80},
    {"obstacle_detected": False, "battery_critical": True},
    {"obstacle_detected": False, "inventory_low": True, "battery_level": 60},
]

print("Hybrid Agent Demo: Warehouse Robot")
print("-" * 50)
for i, scenario in enumerate(scenarios, 1):
    action, layer = hybrid_robot.act(scenario)
    print(f"Scenario {i}: {scenario}")
    print(f"  → Action: {action} (via {layer} layer)")
    print()

Hybrid Agent Demo: Warehouse Robot
--------------------------------------------------
Scenario 1: {'obstacle_detected': True, 'battery_level': 80}
  → Action: emergency_stop (via reactive layer)

Scenario 2: {'obstacle_detected': False, 'battery_critical': True}
  → Action: emergency_return_to_dock (via reactive layer)

Scenario 3: {'obstacle_detected': False, 'inventory_low': True, 'battery_level': 60}
  → Action: check_supplier_availability (via deliberative layer)



### 3.4 Architecture Comparison

| Aspect | Reactive | Deliberative | Hybrid |
|--------|----------|--------------|--------|
| Response time | Fast | Slow | Fast for urgent, planned for others |
| Memory | None | Yes (world model) | Both |
| Planning | None | Multi-step | Strategic + tactical |
| Complexity | Low | Medium | High |
| Best for | Simple, time-critical | Complex, strategic | Real-world systems |

---
## 4. Tools and Action Spaces

Agents become powerful when they can **call external tools**. A tool is any function the agent can invoke to interact with the world.

### 4.1 What Is a Tool?

A **tool** is:
- A well-defined function with inputs and outputs
- An extension of what the agent can do
- Something the agent chooses to call (or not)

Examples of tools:
- `search_database(query)` → returns relevant records
- `send_email(to, subject, body)` → sends an email
- `detect_objects(image)` → returns bounding boxes
- `get_weather(city)` → returns weather data

### 4.2 Tool Specification

For an agent to use a tool, it needs to know:
1. **Name**: What is this tool called?
2. **Description**: What does it do?
3. **Parameters**: What inputs does it need?
4. **Returns**: What output does it produce?

This is the tool's **contract** or **specification**.

In [7]:
@dataclass
class ToolSpec:
    """Specification for a tool that an agent can use."""
    name: str
    description: str
    parameters: Dict[str, str]  # param_name -> description
    returns: str
    function: Callable  # The actual implementation


# Example tools
def count_objects(image_path: str, object_type: str) -> Dict[str, Any]:
    """Simulated object detection tool."""
    # In reality, this would call a YOLO model
    counts = {
        "person": random.randint(0, 20),
        "car": random.randint(0, 10),
        "dog": random.randint(0, 5),
    }
    return {
        "object_type": object_type,
        "count": counts.get(object_type, 0),
        "confidence": 0.95
    }

def send_alert(message: str, priority: str) -> Dict[str, Any]:
    """Simulated alert sending tool."""
    return {
        "status": "sent",
        "message": message,
        "priority": priority,
        "timestamp": "2024-01-15T10:30:00Z"
    }

# Tool specifications
count_objects_spec = ToolSpec(
    name="count_objects",
    description="Count objects of a specific type in an image using computer vision.",
    parameters={
        "image_path": "Path to the image file to analyze",
        "object_type": "Type of object to count (e.g., 'person', 'car', 'dog')"
    },
    returns="Dictionary with object_type, count, and confidence score",
    function=count_objects
)

send_alert_spec = ToolSpec(
    name="send_alert",
    description="Send an alert message to the monitoring system.",
    parameters={
        "message": "The alert message content",
        "priority": "Priority level: 'low', 'medium', 'high', 'critical'"
    },
    returns="Dictionary with status, message, priority, and timestamp",
    function=send_alert
)

print("Tool: count_objects")
print(f"  Description: {count_objects_spec.description}")
print(f"  Parameters: {list(count_objects_spec.parameters.keys())}")

Tool: count_objects
  Description: Count objects of a specific type in an image using computer vision.
  Parameters: ['image_path', 'object_type']


### 4.3 Tool Selection

When an agent has multiple tools available, it must **select** which one to use. This selection can be:

1. **Rule-based**: If condition X, use tool Y.
2. **LLM-based**: Ask the LLM which tool to use given the context.
3. **Learned**: Train a model to select tools.

In LLM-based agents, tool selection is often done by:
1. Providing tool specifications in the prompt.
2. Asking the LLM to output a structured response indicating tool choice.
3. Parsing the response and executing the chosen tool.

In [8]:
class ToolRegistry:
    """Registry of available tools for an agent."""
    
    def __init__(self):
        self.tools: Dict[str, ToolSpec] = {}
    
    def register(self, tool: ToolSpec):
        """Register a tool."""
        self.tools[tool.name] = tool
    
    def get_tool(self, name: str) -> Optional[ToolSpec]:
        """Get a tool by name."""
        return self.tools.get(name)
    
    def execute(self, name: str, **kwargs) -> Any:
        """Execute a tool by name with given arguments."""
        tool = self.get_tool(name)
        if not tool:
            return {"error": f"Unknown tool: {name}"}
        try:
            return tool.function(**kwargs)
        except Exception as e:
            return {"error": str(e)}
    
    def list_tools(self) -> str:
        """Get a formatted list of available tools (for prompts)."""
        lines = []
        for name, spec in self.tools.items():
            params = ", ".join(spec.parameters.keys())
            lines.append(f"- {name}({params}): {spec.description}")
        return "\n".join(lines)


# Create a registry and register our tools
registry = ToolRegistry()
registry.register(count_objects_spec)
registry.register(send_alert_spec)

print("Available tools:")
print(registry.list_tools())
print()

# Execute a tool
result = registry.execute("count_objects", image_path="entrance.jpg", object_type="person")
print(f"Tool execution result: {result}")

Available tools:
- count_objects(image_path, object_type): Count objects of a specific type in an image using computer vision.
- send_alert(message, priority): Send an alert message to the monitoring system.

Tool execution result: {'object_type': 'person', 'count': 20, 'confidence': 0.95}


### 4.4 Tool Failure Handling

Tools can fail. Good agents handle failures gracefully:

1. **Retry**: Try again (perhaps with different parameters).
2. **Fallback**: Use an alternative tool or approach.
3. **Report**: Inform the user or log the error.
4. **Abort**: Stop if the failure is critical.

```python
# Example failure handling pattern
result = registry.execute("some_tool", ...)
if "error" in result:
    # Decide what to do: retry, fallback, report, or abort
    pass
```

---
## 5. Memory

Memory allows agents to maintain context across interactions. There are two main types:

### 5.1 Short-Term Memory (Working Memory)

**Short-term memory** holds information relevant to the current task:
- Recent observations
- Current conversation history
- Intermediate results

**Characteristics:**
- Limited capacity (e.g., last N turns)
- Fast access
- Cleared when task completes

**In LLM agents:** This is typically the conversation history passed to each API call.

In [9]:
@dataclass
class Message:
    """A message in the conversation."""
    role: str  # 'user', 'assistant', 'system', 'tool'
    content: str


class ShortTermMemory:
    """Short-term memory with fixed capacity."""
    
    def __init__(self, max_messages: int = 10):
        self.messages: List[Message] = []
        self.max_messages = max_messages
    
    def add(self, role: str, content: str):
        """Add a message, evicting oldest if at capacity."""
        self.messages.append(Message(role=role, content=content))
        if len(self.messages) > self.max_messages:
            self.messages.pop(0)  # Remove oldest
    
    def get_context(self) -> List[Dict[str, str]]:
        """Get messages in format suitable for LLM."""
        return [{"role": m.role, "content": m.content} for m in self.messages]
    
    def clear(self):
        """Clear all messages."""
        self.messages = []


# Demo
memory = ShortTermMemory(max_messages=5)
memory.add("user", "How many people are in the store?")
memory.add("assistant", "I'll check the entrance camera.")
memory.add("tool", "count_objects returned: {'count': 47}")
memory.add("assistant", "There are currently 47 people in the store.")

print("Short-term memory contents:")
for msg in memory.get_context():
    print(f"  [{msg['role']}]: {msg['content'][:50]}...")

Short-term memory contents:
  [user]: How many people are in the store?...
  [assistant]: I'll check the entrance camera....
  [tool]: count_objects returned: {'count': 47}...
  [assistant]: There are currently 47 people in the store....


### 5.2 Long-Term Memory (Persistent Memory)

**Long-term memory** stores information that persists across sessions:
- User preferences
- Historical data
- Learned patterns
- Knowledge base entries

**Characteristics:**
- Large or unlimited capacity
- Requires retrieval mechanism (search, embeddings)
- Persists across sessions

**Common implementations:**
- Vector databases (for semantic search)
- Key-value stores
- Relational databases
- File storage

In [10]:
class SimpleLongTermMemory:
    """
    Simple long-term memory using keyword matching.
    (In production, you'd use vector embeddings for semantic search.)
    """
    
    def __init__(self):
        self.entries: List[Dict[str, Any]] = []
    
    def store(self, content: str, metadata: Dict[str, Any] = None):
        """Store a memory entry."""
        entry = {
            "content": content,
            "metadata": metadata or {},
            "keywords": set(content.lower().split())
        }
        self.entries.append(entry)
    
    def retrieve(self, query: str, top_k: int = 3) -> List[str]:
        """Retrieve entries matching query keywords."""
        query_keywords = set(query.lower().split())
        
        scored = []
        for entry in self.entries:
            overlap = len(query_keywords & entry["keywords"])
            if overlap > 0:
                scored.append((overlap, entry["content"]))
        
        scored.sort(reverse=True, key=lambda x: x[0])
        return [content for _, content in scored[:top_k]]


# Demo
ltm = SimpleLongTermMemory()
ltm.store("The store typically has peak traffic between 12pm and 2pm.")
ltm.store("Maximum safe capacity for the store is 200 people.")
ltm.store("Emergency exits are located at the back of the store.")
ltm.store("The store manager prefers email alerts for non-critical issues.")

query = "What is the store capacity?"
results = ltm.retrieve(query, top_k=2)

print(f"Query: '{query}'")
print("Retrieved memories:")
for r in results:
    print(f"  - {r}")

Query: 'What is the store capacity?'
Retrieved memories:
  - Maximum safe capacity for the store is 200 people.
  - The store typically has peak traffic between 12pm and 2pm.


### 5.3 Memory Strategies

| Strategy | When to Use |
|----------|-------------|
| **Sliding window** | Keep last N messages; simple but lossy |
| **Summarization** | Compress old messages into summaries |
| **Selective** | Keep only messages marked as important |
| **Hierarchical** | Short-term for recent, long-term for historical |

> **Key Insight**: Memory management is a trade-off between context relevance and capacity limits.

---
## 6. Planning and Control

How does an agent decide what to do? This is the **control** problem.

### 6.1 Simple Control: Condition-Action

The simplest control is reactive rules (as we saw in Section 3.1).

### 6.2 Goal-Based Control

More sophisticated agents work toward explicit **goals**:
1. Define success criteria (goal state).
2. Plan steps to reach the goal.
3. Execute plan, adjusting as needed.
4. Check if goal is achieved.

In [11]:
@dataclass
class Goal:
    """A goal for the agent to achieve."""
    description: str
    success_condition: Callable[[Dict[str, Any]], bool]
    priority: int = 1  # Higher = more important


class GoalBasedController:
    """Controller that manages goals and plans."""
    
    def __init__(self):
        self.goals: List[Goal] = []
        self.current_plan: List[str] = []
    
    def add_goal(self, goal: Goal):
        """Add a goal, maintaining priority order."""
        self.goals.append(goal)
        self.goals.sort(key=lambda g: g.priority, reverse=True)
    
    def get_active_goals(self, state: Dict[str, Any]) -> List[Goal]:
        """Get goals that haven't been achieved yet."""
        return [g for g in self.goals if not g.success_condition(state)]
    
    def replan(self, state: Dict[str, Any]) -> List[str]:
        """
        Create a plan for achieving active goals.
        (Simplified planning logic for illustration.)
        """
        active_goals = self.get_active_goals(state)
        plan = []
        
        for goal in active_goals:
            if "capacity" in goal.description.lower():
                plan.extend(["check_current_count", "compare_to_limit", "alert_if_exceeded"])
            elif "security" in goal.description.lower():
                plan.extend(["scan_for_anomalies", "verify_authorized_personnel"])
        
        self.current_plan = plan
        return plan


# Demo
controller = GoalBasedController()

controller.add_goal(Goal(
    description="Maintain safe capacity levels",
    success_condition=lambda s: s.get("people_count", 0) < s.get("max_capacity", 200),
    priority=2
))

controller.add_goal(Goal(
    description="Ensure security compliance",
    success_condition=lambda s: s.get("security_check_passed", False),
    priority=1
))

state = {"people_count": 180, "max_capacity": 200, "security_check_passed": False}

print("Goal-Based Controller Demo")
print("-" * 40)
active = controller.get_active_goals(state)
print(f"Active goals: {[g.description for g in active]}")

plan = controller.replan(state)
print(f"Generated plan: {plan}")

Goal-Based Controller Demo
----------------------------------------
Active goals: ['Ensure security compliance']
Generated plan: ['scan_for_anomalies', 'verify_authorized_personnel']


### 6.3 ReAct Pattern (Reasoning + Acting)

A popular pattern in LLM agents is **ReAct** (Reason + Act):

1. **Thought**: The agent reasons about what to do.
2. **Action**: The agent chooses a tool/action.
3. **Observation**: The agent receives the result.
4. **Repeat** until done.

```
Thought: I need to find out how many people are in the store.
Action: count_objects(image="entrance_cam.jpg", object_type="person")
Observation: {"count": 187, "confidence": 0.94}

Thought: That's close to capacity (200). I should send an alert.
Action: send_alert(message="Approaching capacity: 187/200", priority="high")
Observation: {"status": "sent"}

Thought: Alert sent. Task complete.
Action: FINISH
```

This pattern makes the agent's reasoning explicit and debuggable.

### 6.4 Stopping Conditions

Agents need to know when to stop. Common stopping conditions:

| Condition | Description |
|-----------|-------------|
| **Goal achieved** | Success criteria met |
| **Max iterations** | Safety limit reached |
| **User request** | User says "stop" or "done" |
| **Error threshold** | Too many failures |
| **Timeout** | Time limit exceeded |
| **Explicit action** | Agent outputs FINISH/DONE |

---
## 7. Multi-Agent Systems

Sometimes one agent isn't enough. **Multi-agent systems** use multiple specialized agents working together.

### 7.1 Why Multiple Agents?

- **Specialization**: Different agents excel at different tasks.
- **Parallelism**: Multiple agents can work simultaneously.
- **Modularity**: Easier to develop, test, and maintain.
- **Robustness**: If one agent fails, others can compensate.

### 7.2 Communication Patterns

**Centralized (Hub-and-Spoke):**
```
        Agent A
           │
           ▼
    ┌──────────────┐
    │ Coordinator  │
    └──────────────┘
      ▲         ▲
      │         │
  Agent B    Agent C
```

**Decentralized (Peer-to-Peer):**
```
  Agent A ◄────► Agent B
      ▲            ▲
      │            │
      └───► Agent C ◄┘
```

**Hierarchical:**
```
         Manager
        /       \
   Supervisor   Supervisor
    /    \        /    \
 Worker Worker Worker Worker
```

In [12]:
class SimpleAgent:
    """A simple agent that can receive and send messages."""
    
    def __init__(self, name: str, specialty: str):
        self.name = name
        self.specialty = specialty
        self.inbox: List[Dict[str, str]] = []
    
    def receive(self, message: str, sender: str):
        """Receive a message."""
        self.inbox.append({"from": sender, "content": message})
    
    def process(self) -> str:
        """Process inbox and generate response."""
        if not self.inbox:
            return "No messages to process."
        
        # Simulate processing based on specialty
        msg = self.inbox.pop(0)
        return f"[{self.name}] Processed '{msg['content'][:30]}...' using {self.specialty} expertise."


class Coordinator:
    """Coordinates multiple agents."""
    
    def __init__(self):
        self.agents: Dict[str, SimpleAgent] = {}
    
    def register(self, agent: SimpleAgent):
        """Register an agent."""
        self.agents[agent.name] = agent
    
    def route(self, task: str) -> str:
        """Route task to appropriate agent based on keywords."""
        task_lower = task.lower()
        
        for name, agent in self.agents.items():
            if agent.specialty.lower() in task_lower:
                agent.receive(task, "coordinator")
                return agent.process()
        
        return "No suitable agent found for this task."


# Demo: Multi-agent store monitoring system
coordinator = Coordinator()

coordinator.register(SimpleAgent("VisionBot", "detection"))
coordinator.register(SimpleAgent("AlertBot", "alert"))
coordinator.register(SimpleAgent("ReportBot", "report"))

tasks = [
    "Run detection on the entrance camera feed.",
    "Send an alert about high traffic.",
    "Generate a daily report of visitor counts.",
]

print("Multi-Agent Coordination Demo")
print("-" * 50)
for task in tasks:
    result = coordinator.route(task)
    print(f"Task: {task}")
    print(f"  → {result}")
    print()

Multi-Agent Coordination Demo
--------------------------------------------------
Task: Run detection on the entrance camera feed.
  → [VisionBot] Processed 'Run detection on the entrance ...' using detection expertise.

Task: Send an alert about high traffic.
  → [AlertBot] Processed 'Send an alert about high traff...' using alert expertise.

Task: Generate a daily report of visitor counts.
  → [ReportBot] Processed 'Generate a daily report of vis...' using report expertise.



### 7.3 When to Use Multi-Agent

| Scenario | Single Agent | Multi-Agent |
|----------|--------------|-------------|
| Simple, linear tasks | ✓ | Overkill |
| Tasks requiring diverse expertise | Limited | ✓ |
| Need for parallel processing | Sequential | ✓ |
| High reliability requirements | Single point of failure | ✓ |
| Complex coordination needed | Simple | ✓ but complex |

> **Warning**: Multi-agent systems add complexity. Start simple; add agents only when needed.

---
## 8. Evaluation and Reliability

How do you know your agent works? This section covers evaluation strategies.

### 8.1 What to Evaluate

| Dimension | Question | Metrics |
|-----------|----------|----------|
| **Correctness** | Does it produce right answers? | Accuracy, F1, exact match |
| **Efficiency** | How many steps/tokens/calls? | Step count, token usage, latency |
| **Robustness** | Does it handle edge cases? | Error rate, recovery rate |
| **Safety** | Does it avoid harmful actions? | Violation rate, guardrail triggers |
| **User satisfaction** | Do users like the results? | Ratings, completion rate |

### 8.2 Test Case Design

Good agent tests include:

1. **Happy path**: Normal scenarios that should succeed.
2. **Edge cases**: Boundary conditions, empty inputs, etc.
3. **Error cases**: Invalid inputs, tool failures, etc.
4. **Adversarial cases**: Attempts to confuse or manipulate the agent.

In [13]:
@dataclass
class TestCase:
    """A test case for agent evaluation."""
    name: str
    input_data: Dict[str, Any]
    expected_action: str
    category: str  # 'happy_path', 'edge_case', 'error_case', 'adversarial'


class AgentEvaluator:
    """Evaluates agent performance on test cases."""
    
    def __init__(self, agent):
        self.agent = agent
        self.results: List[Dict[str, Any]] = []
    
    def run_test(self, test: TestCase) -> Dict[str, Any]:
        """Run a single test case."""
        actual_action = self.agent.act(test.input_data)
        passed = actual_action == test.expected_action
        
        result = {
            "name": test.name,
            "category": test.category,
            "passed": passed,
            "expected": test.expected_action,
            "actual": actual_action
        }
        self.results.append(result)
        return result
    
    def run_suite(self, tests: List[TestCase]) -> Dict[str, Any]:
        """Run all tests and summarize results."""
        for test in tests:
            self.run_test(test)
        
        passed = sum(1 for r in self.results if r["passed"])
        total = len(self.results)
        
        by_category = {}
        for r in self.results:
            cat = r["category"]
            if cat not in by_category:
                by_category[cat] = {"passed": 0, "total": 0}
            by_category[cat]["total"] += 1
            if r["passed"]:
                by_category[cat]["passed"] += 1
        
        return {
            "total_passed": passed,
            "total_tests": total,
            "pass_rate": passed / total if total > 0 else 0,
            "by_category": by_category
        }


# Demo: Test our reactive store agent
test_cases = [
    TestCase("normal_traffic", {"people_count": 30}, "log_normal_traffic", "happy_path"),
    TestCase("high_traffic", {"people_count": 75}, "log_high_traffic", "happy_path"),
    TestCase("crowding", {"people_count": 150}, "send_crowding_alert", "happy_path"),
    TestCase("empty_store", {"people_count": 0}, "no_action", "edge_case"),
    TestCase("boundary_50", {"people_count": 50}, "log_normal_traffic", "edge_case"),
    TestCase("boundary_51", {"people_count": 51}, "log_high_traffic", "edge_case"),
]

evaluator = AgentEvaluator(reactive_agent)  # Using our earlier reactive agent
summary = evaluator.run_suite(test_cases)

print("Agent Evaluation Results")
print("-" * 40)
print(f"Overall: {summary['total_passed']}/{summary['total_tests']} passed ({summary['pass_rate']:.0%})")
print("\nBy category:")
for cat, stats in summary['by_category'].items():
    print(f"  {cat}: {stats['passed']}/{stats['total']}")

Agent Evaluation Results
----------------------------------------
Overall: 6/6 passed (100%)

By category:
  happy_path: 3/3
  edge_case: 3/3


### 8.3 Reliability Patterns

To build reliable agents:

1. **Retry with backoff**: Retry failed operations with increasing delays.
2. **Circuit breaker**: Stop calling a failing service temporarily.
3. **Fallback**: Have backup strategies when primary approach fails.
4. **Logging**: Record all decisions and actions for debugging.
5. **Guardrails**: Validate outputs before acting; block harmful actions.

```python
# Example: Simple retry with backoff
def retry_with_backoff(fn, max_retries=3, base_delay=1.0):
    for attempt in range(max_retries):
        try:
            return fn()
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            delay = base_delay * (2 ** attempt)
            time.sleep(delay)
```

---
## 9. Bridge Back to This Course

Let's connect everything we've learned to the CV models from Part 1.

### 9.1 CV Models as Tools

Your YOLO models from Part 1 are **tools** that an agent can call:

| CV Capability | Tool Interface | Returns |
|---------------|----------------|----------|
| Object Detection | `detect(image)` | Bounding boxes, classes, confidences |
| Segmentation | `segment(image)` | Masks, class labels |
| Pose Estimation | `estimate_pose(image)` | Keypoints, skeleton |
| Tracking | `track(video_frame, tracker_state)` | Track IDs, positions |

The agent decides **when** to call these tools and **what to do** with the results.

In [14]:
# Conceptual example: CV tools for an agent

def detect_tool(image_path: str, model: str = "yolo11n") -> Dict[str, Any]:
    """
    Object detection tool wrapping YOLO.
    
    In reality:
        from ultralytics import YOLO
        model = YOLO('yolo11n.pt')
        results = model(image_path)
        return parse_results(results)
    """
    # Simulated response
    return {
        "detections": [
            {"class": "person", "confidence": 0.92, "bbox": [100, 50, 200, 300]},
            {"class": "person", "confidence": 0.88, "bbox": [250, 60, 350, 310]},
            {"class": "bag", "confidence": 0.75, "bbox": [150, 250, 180, 300]},
        ],
        "image_size": [640, 480],
        "model": model
    }

def count_by_class(detections: List[Dict]) -> Dict[str, int]:
    """Count detections by class."""
    counts = {}
    for det in detections:
        cls = det["class"]
        counts[cls] = counts.get(cls, 0) + 1
    return counts


# Agent using CV tool
print("CV Tool Integration Example")
print("-" * 40)

# Step 1: Agent decides to check camera
print("Agent thought: I should check the entrance camera.")

# Step 2: Agent calls detection tool
result = detect_tool("entrance_cam_frame.jpg")
print(f"Tool called: detect_tool('entrance_cam_frame.jpg')")
print(f"Tool returned: {len(result['detections'])} detections")

# Step 3: Agent processes results
counts = count_by_class(result["detections"])
print(f"Agent analysis: {counts}")

# Step 4: Agent decides next action
if counts.get("person", 0) > 1:
    print("Agent decision: Multiple people detected, log traffic.")

CV Tool Integration Example
----------------------------------------
Agent thought: I should check the entrance camera.
Tool called: detect_tool('entrance_cam_frame.jpg')
Tool returned: 3 detections
Agent analysis: {'person': 2, 'bag': 1}
Agent decision: Multiple people detected, log traffic.


### 9.2 Example: Retail Analytics Agent

Here's a conceptual design for a retail monitoring agent:

```
┌─────────────────────────────────────────────────────┐
│              RETAIL ANALYTICS AGENT                 │
├─────────────────────────────────────────────────────┤
│                                                     │
│  PERCEPTION (CV Tools from Part 1)                  │
│  ├── detect_objects(frame) → boxes, classes         │
│  ├── track_objects(video) → trajectories            │
│  └── estimate_poses(frame) → keypoints              │
│                                                     │
│  REASONING (LLM + Prompt Patterns from Part 2)      │
│  ├── summarize_observations()                       │
│  ├── classify_situation()                           │
│  └── generate_report()                              │
│                                                     │
│  ACTION                                             │
│  ├── send_alert(message, priority)                  │
│  ├── log_event(event_type, data)                    │
│  └── update_dashboard(metrics)                      │
│                                                     │
│  MEMORY                                             │
│  ├── Short-term: recent observations                │
│  └── Long-term: historical patterns, config         │
│                                                     │
└─────────────────────────────────────────────────────┘
```

The CV models **observe**. The LLM **decides**. The tools **act**.

### 9.3 Prompt + Agent Integration

From notebook 09 (LLMs & Prompting), you learned prompt patterns. In an agent context:

| Prompt Pattern | Agent Use |
|----------------|----------|
| **Few-shot** | Show examples of good tool selection |
| **Decomposition** | Break complex goals into steps |
| **Schema enforcement** | Structure tool calls as JSON |
| **Self-check** | Verify action before execution |

Example system prompt for a tool-using agent:

```
You are a retail monitoring assistant. You have access to these tools:
- detect_objects(image): Detect objects in an image
- send_alert(message, priority): Send an alert

When you need to use a tool, respond with:
{"tool": "tool_name", "args": {...}}

When the task is complete, respond with:
{"done": true, "summary": "..."}
```

---
## 10. Exercises

Apply what you've learned with these hands-on exercises.

In [15]:
# Configuration: Set to True to see solutions
SHOW_SOLUTIONS = False

### Exercise 1: Design a Reactive Agent

Create a reactive agent for a parking lot monitoring system.

Rules:
- If `available_spots` < 5: return `"display_full_sign"`
- If `available_spots` < 20: return `"display_limited_sign"`
- Otherwise: return `"display_available_sign"`

In [16]:
# TODO: Define parking lot rules
parking_rules = [
    # (condition_function, action),
    # ...
]

# TODO: Create the reactive agent
# parking_agent = ReactiveAgent(parking_rules)

# Test cases
test_observations = [
    {"available_spots": 3},
    {"available_spots": 15},
    {"available_spots": 50},
]

# TODO: Test the agent
# for obs in test_observations:
#     action = parking_agent.act(obs)
#     print(f"Spots: {obs['available_spots']} → {action}")

In [17]:
if SHOW_SOLUTIONS:
    parking_rules = [
        (lambda obs: obs.get("available_spots", 0) < 5, "display_full_sign"),
        (lambda obs: obs.get("available_spots", 0) < 20, "display_limited_sign"),
        (lambda obs: True, "display_available_sign"),
    ]
    
    parking_agent = ReactiveAgent(parking_rules)
    
    print("Solution: Parking Lot Agent")
    print("-" * 40)
    for obs in test_observations:
        action = parking_agent.act(obs)
        print(f"Spots: {obs['available_spots']:2d} → {action}")

### Exercise 2: Implement a Tool Registry

Create tools for a security monitoring system and register them.

Tools to implement:
1. `check_motion(zone)` - Returns whether motion is detected in a zone
2. `sound_alarm(level)` - Sounds an alarm at specified level
3. `notify_security(message)` - Sends a message to security personnel

In [18]:
# TODO: Implement the tool functions
def check_motion(zone: str) -> Dict[str, Any]:
    """Check for motion in a security zone."""
    # Hint: Return {"zone": zone, "motion_detected": True/False}
    pass

def sound_alarm(level: str) -> Dict[str, Any]:
    """Sound an alarm at the specified level."""
    # Hint: Return {"status": "activated", "level": level}
    pass

def notify_security(message: str) -> Dict[str, Any]:
    """Notify security personnel."""
    # Hint: Return {"status": "sent", "message": message}
    pass

# TODO: Create ToolSpec for each tool
# TODO: Create a ToolRegistry and register the tools
# TODO: Test executing each tool

In [19]:
if SHOW_SOLUTIONS:
    def check_motion(zone: str) -> Dict[str, Any]:
        """Check for motion in a security zone."""
        return {
            "zone": zone,
            "motion_detected": random.choice([True, False]),
            "timestamp": "2024-01-15T22:30:00Z"
        }
    
    def sound_alarm(level: str) -> Dict[str, Any]:
        """Sound an alarm at the specified level."""
        return {
            "status": "activated",
            "level": level,
            "duration_seconds": 30 if level == "high" else 10
        }
    
    def notify_security(message: str) -> Dict[str, Any]:
        """Notify security personnel."""
        return {
            "status": "sent",
            "message": message,
            "recipients": ["guard_1", "guard_2"]
        }
    
    # Create tool specs
    motion_spec = ToolSpec(
        name="check_motion",
        description="Check for motion in a security zone.",
        parameters={"zone": "Name of the zone to check"},
        returns="Dict with zone, motion_detected, timestamp",
        function=check_motion
    )
    
    alarm_spec = ToolSpec(
        name="sound_alarm",
        description="Sound an alarm at specified level.",
        parameters={"level": "Alarm level: 'low', 'medium', 'high'"},
        returns="Dict with status, level, duration",
        function=sound_alarm
    )
    
    notify_spec = ToolSpec(
        name="notify_security",
        description="Send a message to security personnel.",
        parameters={"message": "The message to send"},
        returns="Dict with status, message, recipients",
        function=notify_security
    )
    
    # Create and populate registry
    security_registry = ToolRegistry()
    security_registry.register(motion_spec)
    security_registry.register(alarm_spec)
    security_registry.register(notify_spec)
    
    print("Solution: Security Tool Registry")
    print("-" * 40)
    print(security_registry.list_tools())
    print()
    print("Test executions:")
    print(security_registry.execute("check_motion", zone="entrance"))
    print(security_registry.execute("sound_alarm", level="high"))
    print(security_registry.execute("notify_security", message="Motion detected in Zone A"))

### Exercise 3: Design Test Cases

Design a test suite for the parking lot agent from Exercise 1.

Include:
- At least 3 happy path cases
- At least 2 edge cases (boundary conditions)
- At least 1 error case

In [20]:
# TODO: Create test cases for the parking lot agent
parking_test_cases = [
    # Happy path cases
    # TestCase("name", {"available_spots": X}, "expected_action", "happy_path"),
    
    # Edge cases
    # TestCase("name", {"available_spots": X}, "expected_action", "edge_case"),
    
    # Error cases
    # TestCase("name", {...}, "expected_action", "error_case"),
]

# TODO: Run the test suite using AgentEvaluator
# evaluator = AgentEvaluator(parking_agent)
# results = evaluator.run_suite(parking_test_cases)
# print(results)

In [21]:
if SHOW_SOLUTIONS:
    # First, ensure we have the parking agent from Exercise 1
    parking_rules = [
        (lambda obs: obs.get("available_spots", 0) < 5, "display_full_sign"),
        (lambda obs: obs.get("available_spots", 0) < 20, "display_limited_sign"),
        (lambda obs: True, "display_available_sign"),
    ]
    parking_agent = ReactiveAgent(parking_rules)
    
    parking_test_cases = [
        # Happy path cases
        TestCase("lot_full", {"available_spots": 2}, "display_full_sign", "happy_path"),
        TestCase("lot_limited", {"available_spots": 10}, "display_limited_sign", "happy_path"),
        TestCase("lot_available", {"available_spots": 100}, "display_available_sign", "happy_path"),
        
        # Edge cases (boundary conditions)
        TestCase("boundary_4_spots", {"available_spots": 4}, "display_full_sign", "edge_case"),
        TestCase("boundary_5_spots", {"available_spots": 5}, "display_limited_sign", "edge_case"),
        TestCase("boundary_19_spots", {"available_spots": 19}, "display_limited_sign", "edge_case"),
        TestCase("boundary_20_spots", {"available_spots": 20}, "display_available_sign", "edge_case"),
        TestCase("zero_spots", {"available_spots": 0}, "display_full_sign", "edge_case"),
        
        # Error case (missing data - falls to default)
        TestCase("missing_data", {}, "display_full_sign", "error_case"),  # defaults to 0
    ]
    
    evaluator = AgentEvaluator(parking_agent)
    results = evaluator.run_suite(parking_test_cases)
    
    print("Solution: Parking Agent Test Suite")
    print("-" * 40)
    print(f"Overall: {results['total_passed']}/{results['total_tests']} passed ({results['pass_rate']:.0%})")
    print("\nBy category:")
    for cat, stats in results['by_category'].items():
        print(f"  {cat}: {stats['passed']}/{stats['total']}")
    
    print("\nFailed tests:")
    for r in evaluator.results:
        if not r["passed"]:
            print(f"  {r['name']}: expected '{r['expected']}', got '{r['actual']}'")

---
## 11. Recap

### Key Takeaways

1. **Agent Definition**: An agent is a system that observes, decides, acts, and updates in a loop.

2. **Architectures**:
   - **Reactive**: Fast, rule-based, no memory
   - **Deliberative**: Plans ahead, maintains world model
   - **Hybrid**: Combines both for real-world robustness

3. **Tools**: Agents extend their capabilities by calling tools. Good tool specs include name, description, parameters, and return type.

4. **Memory**: Short-term for current context, long-term for persistent knowledge. Manage memory carefully due to capacity limits.

5. **Planning**: From simple condition-action rules to goal-based planning and ReAct patterns.

6. **Multi-Agent**: Use when you need specialization, parallelism, or robustness. Adds complexity.

7. **Evaluation**: Test agents like software—happy paths, edge cases, error handling. Measure correctness, efficiency, and robustness.

8. **CV Integration**: Your YOLO models from Part 1 become perception tools that agents can call.

### Checklist

- [ ] I can explain the agent loop (observe → decide → act → update).
- [ ] I understand when to use reactive vs. deliberative vs. hybrid architectures.
- [ ] I can design a tool specification with clear inputs and outputs.
- [ ] I understand the difference between short-term and long-term memory.
- [ ] I can describe the ReAct pattern and when to use it.
- [ ] I know how to design test cases for agent evaluation.
- [ ] I can envision how to integrate CV models as agent tools.

### Next Steps

With the conceptual foundations from this notebook, you're ready to:
- Explore agent frameworks (LangChain, CrewAI, AutoGen) with understanding.
- Build your own simple agents using the patterns learned here.
- Combine your CV models with agent architectures for real applications.

---

*End of Part 2: AI Agents*